# พิมพ์แจ้งเหตุเอง · Agent จริง + Gemini จริง

รันเซลล์ด้านล่างจากบนลงล่าง แล้วพิมพ์ในกล่องแชตและกด **ส่งข้อความ** คุยต่อได้เรื่อย ๆ โดยไม่ต้องรันเซลล์ซ้ำ

- ใช้ FloodAgent และกฎชุดเดียวกับ API มี state, consent, priority และ outbox จริง
- เรียก Gemini จริงเมื่อจำเป็น ใช้ key จาก `dspy-agent/.env` หรือ `lab/1/.env` และมีค่าใช้จ่ายตามการเรียก
- กด Quick Reply ที่ผู้ช่วยแสดงได้ ปุ่มแชร์ตำแหน่งให้กรอกพิกัดสมมติเอง
- กด **ดูข้อมูลที่ระบบเก็บ** เพื่อตรวจ state กับรายงานที่เตรียมส่ง
- กด **เริ่มแชตใหม่** เพื่อเคลียร์บทสนทนาทดลองและ outbox จำลอง
- ยังไม่ต่อ LINE หรือปลายทาง Agent 2 จึงไม่อ้างว่าส่งถึงเจ้าหน้าที่แล้ว

ใช้ข้อมูลสมมติเท่านั้น: Notebook นี้เปิด KB PREVIEW ให้ลองตอบจากเนื้อหาร่าง พร้อมป้ายยังไม่ผ่าน Human Review ในแหล่งข้อมูล ไม่เปลี่ยน CSV และไม่เปิดใช้เนื้อหาร่างใน API/LINE จริง พื้นที่ยังทำงานตามข้อจำกัดเดิม

หาก kernel ยังไม่มี dependency ให้รันคำสั่งนี้ในเซลล์ใหม่ แล้ว Restart Kernel:
```python
%pip install -r /Users/bubboo/DSI490/dspy-agent/requirements-notebook.txt
```


In [ ]:
import os
import sys
from pathlib import Path
import dspy
from dotenv import load_dotenv, dotenv_values

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dspy-agent").is_dir())
APP_ROOT = ROOT / "dspy-agent"
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))
load_dotenv(APP_ROOT / ".env", override=False)
load_dotenv(ROOT / "lab" / "1" / ".env", override=False)  # รองรับไฟล์ key เดิมของ Notebook


In [ ]:
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or os.getenv("API_KEY")
# Read model from disk on every setup run, avoiding stale kernel environment.
model = (dotenv_values(ROOT / "lab" / "1" / ".env").get("GEMINI_MODEL")
         or dotenv_values(APP_ROOT / ".env").get("GEMINI_MODEL")
         or os.getenv("GEMINI_MODEL") or "gemini/gemini-2.5-flash-lite").strip()
# DSPy/LiteLLM needs the provider prefix even when .env contains a bare Gemini model.
if "/" not in model:
    model = "gemini/" + model
if not api_key:
    raise RuntimeError("ตั้ง GEMINI_API_KEY ใน dspy-agent/.env หรือ lab/1/.env ก่อนเรียก Gemini")
lm = dspy.LM(model=model, api_key=api_key, cache=False, timeout=60, num_retries=2)
dspy.configure(lm=lm)
print("model =", model)


In [ ]:
import json
from app.agent import FloodAgent
from app.schemas import Hotline, SCHEMA_VERSION
from app.repository import Repository
from app.knowledge import KnowledgeBase
from app.areas import AreaResolver

# โหลด config เดียวกับ API; ห้ามสร้าง verified_at ใหม่เพื่อแสร้งว่าตรวจแล้ว
hotline_config = json.loads((APP_ROOT / "app" / "hotlines_th.json").read_text(encoding="utf-8"))
HOTLINES = [Hotline.model_validate(row) for row in hotline_config["hotlines"]]
print("schema =", SCHEMA_VERSION, "| hotline review =", hotline_config.get("status"))


class PreviewKnowledgeBase(KnowledgeBase):
    """Draft evidence for this notebook only; no approval records are modified."""
    def _eligible(self, row, now):
        if row.get("review_status") != "pending_review":
            return super()._eligible(row, now)
        if row.get("category") != "A" or not all(row.get(k) for k in (
            "chunk_id", "source_org", "title", "url", "reviewed_text", "source_checked_at"
        )):
            return False
        from datetime import datetime
        try:
            checked = datetime.fromisoformat(row["source_checked_at"])
            if checked.tzinfo is None or checked > now:
                return False
            if row.get("valid_until"):
                expiry = datetime.fromisoformat(row["valid_until"])
                if expiry.tzinfo is None or expiry <= now:
                    return False
        except ValueError:
            return False
        return True

    @staticmethod
    def _evidence(row):
        from dataclasses import replace
        evidence = KnowledgeBase._evidence(row)
        if row.get("review_status") == "pending_review":
            return replace(evidence, title="[ทดลอง: ยังไม่ผ่าน Human Review] " + evidence.title)
        return evidence


print("KB PREVIEW: เนื้อหาร่างสำหรับข้อมูลสมมติเท่านั้น")


def make_agent():
    # Notebook จำลองการส่งคำตอบสำเร็จ และแยกข้อมูลจากฐานข้อมูล API
    # ไม่มี sender: รายงานอยู่ใน outbox เป็น pending ไม่ส่ง network ไป Agent 2
    return FloodAgent(
        hotlines=HOTLINES,
        repository=Repository("sqlite:///:memory:"),
        knowledge=PreviewKnowledgeBase(APP_ROOT / "app" / "knowledge_th.csv"),
        area_resolver=AreaResolver(APP_ROOT / "app" / "areas_singburi.csv"),
        internal_user_id="usr_notebook_demo",
    )


In [ ]:
# รันครั้งเดียว แล้วพิมพ์ในกล่องด้านล่างได้เลย
notebook_dir = ROOT / "lab" / "1"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
from notebook_chat import NotebookChat

chat = NotebookChat(make_agent)
chat.display();
